# Exploratory Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import anderson,normaltest
import duckdb
from pingouin.correlation import pearsonr,spearmanr
import pingouin as pg
from scipy.stats import pointbiserialr,levene
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from patsy.builtins import Q
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import scikit_posthocs as sp
from sklearn.feature_selection import mutual_info_regression
import sys
sys.path.append('..\src')
from py_def_class import games_howell_simple_effects,robust_z_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_csv(r'..\data\processed\Cleaned_quali_f1.csv')
df.head()

,Driver,Team,GP,Rule_Era,AirTemp_p1,Humidity_p1,Pressure_p1,Rainfall_p1,TrackTemp_p1,laptime_sum_sectortimes_p1,LapTimeDiff_p1,Compound_p1,AirTemp_p2,Humidity_p2,Pressure_p2,Rainfall_p2,TrackTemp_p2,laptime_sum_sectortimes_p2,LapTimeDiff_p2,Compound_p2,AirTemp_p3,Humidity_p3,Pressure_p3,Rainfall_p3,TrackTemp_p3,laptime_sum_sectortimes_p3,LapTimeDiff_p3,Compound_p3,Sprint-Session,AirTemp_sprint_quali,Humidity_sprint_quali,Pressure_sprint_quali,Rainfall_sprint_quali,TrackTemp_sprint_quali,laptime_sum_sectortimes_sprint_quali,LapTimeDiff_sprint_quali,Compound_sprint_quali,Session,AirTemp_quali,Humidity_quali,Pressure_quali,Rainfall_quali,TrackTemp_quali,laptime_sum_sectortimes_quali,LapTimeDiff_quali,Compound_quali,Season,Sprint_Race_Era,Sprint_Weekend,race_round,gp_id,Team_Lineage,Type,Direction,Circut_length,Turns,Pace_profile,Flat_out_run,Slow_turns,Medium_turns,High_speed_turns,Turn_density,Complexity_label,Qualifying_Date
0,VER,Red Bull Racing,Abu Dhabi GP,High-Downforce,29.7,33.0,1015.7,0.0,41.3,98.491,0.000,HYPERSOFT,26.2,58.4,1016.1,0.0,28.9,97.280,0.044,HYPERSOFT,31.9,29.5,1015.8,0.0,37.4,97.747,0.571,HYPERSOFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q3,28.7,44.0,1016.7,0.0,31.0,95.589,0.795,HYPERSOFT,2018,2023-2026,0,21.0,20,Red Bull Racing,Race,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced,2018-11-24
1,RIC,Red Bull Racing,Abu Dhabi GP,High-Downforce,29.9,29.4,1015.5,0.0,38.2,98.945,0.454,HYPERSOFT,26.1,58.6,1016.1,0.0,29.0,97.428,0.192,HYPERSOFT,32.1,29.6,1015.7,0.0,37.8,98.090,0.914,HYPERSOFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q3,28.5,43.1,1016.7,0.0,30.7,95.401,0.607,HYPERSOFT,2018,2023-2026,0,21.0,20,Red Bull Racing,Race,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced,2018-11-24
2,BOT,Mercedes,Abu Dhabi GP,High-Downforce,29.7,32.6,1015.7,0.0,40.4,99.452,0.961,SUPERSOFT,26.2,57.1,1015.9,0.0,29.4,97.236,0.000,HYPERSOFT,32.1,30.0,1015.7,0.0,38.1,97.933,0.757,HYPERSOFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q3,28.5,43.1,1016.7,0.0,30.7,94.956,0.162,HYPERSOFT,2018,2023-2026,0,21.0,20,Mercedes,Race,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced,2018-11-24
3,HAM,Mercedes,Abu Dhabi GP,High-Downforce,29.6,33.2,1015.8,0.0,43.3,99.543,1.052,ULTRASOFT,26.2,57.1,1015.9,0.0,29.4,97.443,0.207,HYPERSOFT,32.1,30.0,1015.7,0.0,38.1,97.176,0.000,HYPERSOFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q3,28.5,43.1,1016.7,0.0,30.7,94.794,0.000,HYPERSOFT,2018,2023-2026,0,21.0,20,Mercedes,Race,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced,2018-11-24
4,OCO,Racing Point,Abu Dhabi GP,High-Downforce,29.7,32.8,1015.8,0.0,41.0,100.102,1.611,HYPERSOFT,26.2,57.4,1016.0,0.0,29.4,98.402,1.166,HYPERSOFT,31.9,30.0,1015.7,0.0,37.6,99.011,1.835,HYPERSOFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q3,28.5,43.1,1016.7,0.0,30.7,96.540,1.746,HYPERSOFT,2018,2023-2026,0,21.0,20,Force India-Racing Point-Aston Martin,Race,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced,2018-11-24


In [3]:
#logic check for the new 
duckdb.sql("""
    Select
        Driver,
        Team,
        -- gp_id,
        GP,
        Season,
        Session,
        laptime_sum_sectortimes_quali,
        LapTimeDiff_quali
    from df
    where GP = 'Madrid GP'
    and Season = 2026
    -- and laptime_sum_sectortimes_quali is not null
    order by LapTimeDiff_quali asc
""").df()

,Driver,Team,GP,Season,Session,laptime_sum_sectortimes_quali,LapTimeDiff_quali
0,NOR,McLaren,Madrid GP,2026,Q3,91.824,0.000
1,ANT,Mercedes,Madrid GP,2026,Q3,91.835,0.011
2,VER,Red Bull Racing,Madrid GP,2026,Q3,91.964,0.140
3,HAM,Ferrari,Madrid GP,2026,Q3,92.013,0.189
4,LEC,Ferrari,Madrid GP,2026,Q3,92.019,0.195
5,RUS,Mercedes,Madrid GP,2026,Q3,92.149,0.325
6,PIA,McLaren,Madrid GP,2026,Q3,92.294,0.470
7,LAW,Red Bull Racing,Madrid GP,2026,Q3,92.316,0.492
8,HUL,Audi,Madrid GP,2026,Q2,93.223,0.792
9,BOR,Audi,Madrid GP,2026,Q2,93.388,0.957


In [ ]:
df[df['GP'].eq('Australian GP')]['Season'].unique()

In [ ]:
df['gp_id'].unique()

In [ ]:
duckdb.sql("""
    Select
        Distinct Season,
        GP,
        gp_id
    from df
    where Season = 2026
    and GP = 'Monaco GP'
""").df()

In [ ]:
test = duckdb.sql("""
    WITH gp_w_null AS (
        SELECT
            GP,
            Season,
            COUNT(*) AS number_of_appearance
        FROM df
        WHERE laptime_sum_sectortimes_quali IS NULL
        GROUP BY
            GP, Season
    ),

    length_of_gp AS (
        SELECT
            GP,
            Season,
            COUNT(*) AS overall_driver_number_for_gp
        FROM df
        GROUP BY
            GP, Season
    ),

    final AS (
        SELECT
            g.GP,
            g.Season,
            g.number_of_appearance,
            l.overall_driver_number_for_gp,
            CASE
                WHEN 
                    (g.Season = 2026 AND l.overall_driver_number_for_gp = 22)
                    OR 
                    (g.Season <= 2025 AND l.overall_driver_number_for_gp = 20)
                THEN 'watch'
                ELSE 'dont watch'
            END AS status
        FROM gp_w_null AS g
        INNER JOIN length_of_gp AS l
            ON g.GP = l.GP
            AND g.Season = l.Season
    )

    SELECT
        *
    FROM final
    -- where status = 'watch'
""").df()
test

## Checking for NaN Values

Note: We are aware that there a few columns and rows contain NaN values. This is due to sessions are missing (not every race weekend is sprint-race weekend), or drivers did not drive all sessions.

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum()/len(df))*100

As already addressed, we will have NaN values, but it is in a managable range.

## Appearance Count of the Values of each Column

In [ ]:
df.info()

### Appearances of Object Value

In [ ]:
df_ob = df.select_dtypes(exclude='number')
cols_ = df_ob.columns.to_list()

for c in cols_:
    print(df[c].value_counts())
    print('\n')
for c in cols_:
    print((df[c].value_counts()/len(df))*100)
    print('\n')

Here we can see several things:
- we have several drivers with a very low or only one appearance, therefore we will bin all the drivers, which have a lower appearance count then 10,with RareLabel encoder. For all the other driver we will perform a smoothed-mean encoding.
- 2026 a new F1 rule set was introduced, which had an massiv impact on the aerodynamics, power-units and battery. There have been only five races under the new rules yet, therefore we will compare the already completed races of the 2026 season with the previous seasons.
- Every tyer compound, which has a lower or equal count to 10 will be binned into one group (RareLabel encoder), this will be done for the columns/features "compound_p1","compound_p2","compound_p3".
- Even though we have a couple of GPs which only appear once, during the 2020 season, due to COVID-19. Therefore we apply to the GP column two encoders. First, count-frequency, to showcase GPs with a lower appearance, after that, we apply the mean-encoder to track for each race track the mean y performance. Note, we will create a custome class to add two more of the GP columns, so we can encode them.
- For the original Team column, we will use frequency encoding and, if used, smoothed mean encoding. For the Team_Lineage column, we will use both frequency encoding and smoothed mean encoding, similar to the GP feature.

#### Checking on the avergae difference to the last season before the rule change

In [ ]:
gps_ = tuple(df[df['Season'].eq(2026)]['GP'].unique())

comp_races = duckdb.sql(f"""
    select
        *
    from df
    where GP in {gps_}
""").df()

for r in comp_races['GP'].unique():
    plt.figure(figsize=(12,6))
    comp_races['Season'] = comp_races['Season'].astype(str)
    sns.boxplot(data=comp_races[comp_races['GP'].eq(r)],y='laptime_sum_sectortimes_quali',x='Season',hue='Season',palette='Spectral')
    plt.title(f'Quali Time Dev from 2018 - 2026 for {r}')

In [ ]:
#gp_compare = ['Chinese GP', 'Australian GP', 'Japanese GP']
#here we get the average difference of the 2026 to 2025 (of all completed races yet)
gp_compare = df['GP'].unique()
compare_fast = pd.DataFrame()
for g in gp_compare:
    gp_ = df[(df['GP'].eq(g))].groupby(
        ['Season','GP'])['laptime_sum_sectortimes_quali'].min().reset_index().sort_values(['Season'])
    gp_['difference_last_season'] = gp_['laptime_sum_sectortimes_quali'].diff()
    compare_fast = pd.concat([compare_fast,gp_],axis=0)
compare_fast[compare_fast['Season'].eq(2026)]['difference_last_season'].mean()

Based on that outcome, we can say that, under the current rule set, the cars are currently on average 2 seconds slower than the cars from the previous season, 2025, which ran under a different rule set. Based on that information, we will train the model with data from 2018-2025, and make predictions for 2026

## Descriptive Analysis - Object Type
### Pole Postion by Driver and Teams

In [ ]:
df_pole = df[(df['LapTimeDiff_quali']<=0.0) & (df['Session'].eq('Q3'))]
df_pole["Season"] = df_pole["Season"].astype(str)

print(df_pole['Driver'].value_counts())
print('\n')
plt.figure(figsize=(12,6))
df_pole['Driver'].value_counts().plot(kind='bar')
plt.title('Pole Positions from 2018 - 2026',size=18,fontweight='bold')

plt.figure(figsize=(12,6))
sns.countplot(data=df_pole,x='Driver',hue='Season',palette='viridis')

Here we get an overview of all pole-position setters from 2018 to 2026. Next, we will analyze how dominant each pole position was by looking at the gap between the pole sitter and the next-fastest driver. This gives us an indication of which drivers had dominant periods and in which seasons those dominant periods occurred.

This information is relevant for the offset approach. For the 2026 season, the model may make systematic prediction errors because of the new rule set. Therefore, after generating base predictions, we will calculate residuals on the already completed 2026 races. These residuals can then be analyzed by driver, team, and GP.

If enough data is available, we can use smoothed driver-level, team-level, and GP-level offsets to adjust future 2026 predictions. However, because only a small number of 2026 races are available, these offsets should be regularized or weighted by the number of observations to avoid overfitting. For very sparse driver-team-GP combinations, we should fall back to broader offsets such as the global 2026 offset, team offset, or GP offset.

Next, we will check how dominant the pole positions where of each driver, who was able to get a pole-position whitin the period of 2018-2026.

In [ ]:
#getting the differences between pole setter and P2
pps = list(df_pole['Driver'].unique())
for p in pps:
    gp_season_pole = df[(df['Driver'].eq(p)) & (df['LapTimeDiff_quali'].eq(0.0)) & (df['Session'].eq('Q3'))][['GP','Season']].drop_duplicates()
    season_gp_lst = []
    for i in range(0,len(gp_season_pole)):
        season_gp_lst.append((gp_season_pole.iloc[i,0],int(gp_season_pole.iloc[i,1])))
    df_pole_stack = pd.DataFrame()

    #here we get only the second fastest time and stack them based on GP and Season based on wheter the driver got a pole or not
    for sg in season_gp_lst:
        df_loop = df[(df['GP'].eq(sg[0])) & (df['Season'].eq(sg[1])) & (df['Session'].eq('Q3')) & (df['LapTimeDiff_quali']>0.0)]
        p2 = df_loop['LapTimeDiff_quali'].min()
        df_loop = df_loop[df_loop['LapTimeDiff_quali'].eq(p2)]
        df_pole_stack = pd.concat([df_pole_stack,df_loop],axis=0)

    for s in df_pole_stack['Season'].unique():
        print(f'{p}-{s}')
        print(df_pole_stack[df_pole_stack['Season'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Pace_profile'].unique():
        print(f'{p}-{s}')
        print(df_pole_stack[df_pole_stack['Pace_profile'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Complexity_label'].unique():
        print(f'{p}-{s}')
        print(df_pole_stack[df_pole_stack['Complexity_label'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Direction'].unique():
        print(f'{p}-{s}')
        print(df_pole_stack[df_pole_stack['Direction'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    df_pole_stack['Season'] = df_pole_stack['Season'].astype(str)
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',hue='Season',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {p} to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali', x='Season', hue='Pace_profile',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {p} at different track types to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',x='Season',hue='Complexity_label',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {p} at different complexity to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',x='Season',hue='Direction',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {p} at different direction to P2')

Here we can see, that certain driver had a dominant period in time. This becomes obvious, when we focus on the margins to P2. Encoding the drivers can provide some valuable insight in predicting quali lap times.

Next, we will check how the P1-to-P2 margin behaves between teams.

In [ ]:
df_pole = df[(df['LapTimeDiff_quali']<=0.0) & (df['Session'].eq('Q3'))]
df_pole["Season"] = df_pole["Season"].astype(str)

print(df_pole['Team'].value_counts())
print('\n')
plt.figure(figsize=(12,6))
df_pole['Team'].value_counts().plot(kind='bar')
plt.title('Pole Positions from 2018 - 2026',size=18,fontweight='bold')

plt.figure(figsize=(12,6))
sns.countplot(data=df_pole,x='Team',hue='Season',palette='viridis')

In [ ]:
#getting the difference between P1 and P2
pps = list(df_pole['Team'].unique())
for t in pps:
    gp_season_pole = df[(df['Team'].eq(t)) & (df['LapTimeDiff_quali'].eq(0.0)) & (df['Session'].eq('Q3'))][['GP','Season']].drop_duplicates()
    season_gp_lst = []
    for i in range(0,len(gp_season_pole)):
        season_gp_lst.append((gp_season_pole.iloc[i,0],int(gp_season_pole.iloc[i,1])))
    df_pole_stack = pd.DataFrame()

    #here we get only the second fastest time and stack them based on GP and Season based on wheter the team got a pole or not
    for sg in season_gp_lst:
        df_loop = df[(~df['Team'].eq(t)) & (df['GP'].eq(sg[0])) & (df['Season'].eq(sg[1])) & (df['Session'].eq('Q3')) & (df['LapTimeDiff_quali']>0.0)]
        p2 = df_loop['LapTimeDiff_quali'].min()
        df_loop = df_loop[df_loop['LapTimeDiff_quali'].eq(p2)]
        df_pole_stack = pd.concat([df_pole_stack,df_loop],axis=0)
    
    for s in df_pole_stack['Season'].unique():
        print(f'{t}-{s}')
        print(df_pole_stack[df_pole_stack['Season'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Pace_profile'].unique():
        print(f'{t}-{s}')
        print(df_pole_stack[df_pole_stack['Pace_profile'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Complexity_label'].unique():
        print(f'{t}-{s}')
        print(df_pole_stack[df_pole_stack['Complexity_label'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_pole_stack['Direction'].unique():
        print(f'{t}-{s}')
        print(df_pole_stack[df_pole_stack['Direction'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    df_pole_stack['Season'] = df_pole_stack['Season'].astype(str)
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',hue='Season',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {t} to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali', x='Season', hue='Pace_profile',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {t} at different track types to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',x='Season',hue='Complexity_label',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {t} at different complexity to P2')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_pole_stack,y='LapTimeDiff_quali',x='Season',hue='Direction',palette='viridis')
    plt.title(f'Boxeplot of time difference of the pole positions of {t} at different direction to P2')

Here we even see a clearer picture, as with the drivers. In F1, a competetive car, is more important than having a worldclass driver. When you have a bad car a great driver can nowdays do only so much. Here we can see also a clear pattern, teams declining or raising.

Next, we will focus on every driver's and team's lap time development.

### Overall Lap Time Differences by Driver and Teams
#### Driver

In [ ]:
pps = list(df[~df['LapTimeDiff_quali'].isna()]['Driver'].unique())
for p in pps:
    df_driver = df[(df['Driver'].eq(p)) & (~df['LapTimeDiff_quali'].isna()) & (df['LapTimeDiff_quali']<5.0)].copy()

    for s in df_driver['Season'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Season'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Pace_profile'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Pace_profile'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Complexity_label'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Complexity_label'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Direction'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Direction'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    df_driver['Season'] = df_driver['Season'].astype(str)
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',x='Team',hue='Season',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p}')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali', x='Season', hue='Pace_profile',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different track types')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',x='Season',hue='Complexity_label',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different complexity')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',x='Season',hue='Direction',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different direction')

#### Teams

In [ ]:
pps = list(df[~df['LapTimeDiff_quali'].isna()]['Team'].unique())
for p in pps:
    df_driver = df[(df['Team'].eq(p)) & (~df['LapTimeDiff_quali'].isna()) & (df['LapTimeDiff_quali']<5.0)].copy()

    for s in df_driver['Season'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Season'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Pace_profile'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Pace_profile'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Complexity_label'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Complexity_label'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    for s in df_driver['Direction'].unique():
        print(f'{p}-{s}')
        print(df_driver[df_driver['Direction'].eq(s)]['LapTimeDiff_quali'].describe())
        print('\n')

    df_driver['Season'] = df_driver['Season'].astype(str)
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',hue='Season',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p}')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali', x='Season', hue='Pace_profile',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different track types')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',x='Season',hue='Complexity_label',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different complexity')

    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_driver,y='LapTimeDiff_quali',x='Season',hue='Direction',palette='viridis')
    plt.title(f'Boxeplot of lap times of {p} at different direction')

Here we can see that the lap time quali difference differs from season to season, team to team, track to track etc., we can also see that we have a regime shift in 2026, the power structure has changed. These aspect supports the offset approach, in which we will train on 2018-2025 data and predict on 2026 data, then we will use the residuals to get an idea how much off we are on a global, driver, GP and Team level.

### Compound Behaviour
Next, we will check the behaviour of the different tyre compound.

In [ ]:
session_specs = [
    ("P1", "Compound_p1", "laptime_sum_sectortimes_p1"),
    ("P2", "Compound_p2", "laptime_sum_sectortimes_p2"),
    ("P3", "Compound_p3", "laptime_sum_sectortimes_p3"),
    ("Sprint-Quali", "Compound_sprint_quali", "laptime_sum_sectortimes_sprint_quali"),
    ("Quali", "Compound_quali", "laptime_sum_sectortimes_quali"),
]

df_comp_laps = []

for session_name, compound_col, lap_col in session_specs:
    tmp = (
        df.groupby(["Driver", "Team", "GP", "Season", compound_col], dropna=False)[lap_col]
          .min()
          .reset_index()
          .rename(columns={
              compound_col: "Compound",
              lap_col: "LapTime"
          })
    )

    tmp["Session"] = session_name
    df_comp_laps.append(tmp)

df_comp_lap = pd.concat(df_comp_laps, ignore_index=True)

df_comp_lap = df_comp_lap.dropna(subset=["LapTime", "Compound"])

#just visualization
for season in sorted(df_comp_lap["Season"].unique()):
    for gp in df_comp_lap[df_comp_lap["Season"].eq(season)]["GP"].unique():

        df_s_gp = df_comp_lap[
            (df_comp_lap["Season"].eq(season)) &
            (df_comp_lap["GP"].eq(gp))
        ]
        plt.figure(figsize=(12, 6))

        sns.boxplot(
            data=df_s_gp,
            x="Session",
            y="LapTime",
            hue="Compound"
        )

        plt.title(f"{gp} {season} - Best lap time by compound and session")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
#pure numbers
for season in sorted(df_comp_lap["Season"].unique()):
    for gp in df_comp_lap[df_comp_lap["Season"].eq(season)]["GP"].unique():

        df_s_gp = df_comp_lap[
            (df_comp_lap["Season"].eq(season)) &
            (df_comp_lap["GP"].eq(gp))
        ]

        if df_s_gp.empty:
            continue
        
        for c in df_s_gp['Compound'].unique():
            print(f'{gp}-{season}-{c}')
            print(df_s_gp[df_s_gp['Compound'].eq(c)].groupby('Session')['LapTime'].describe())
            print('\n')

In [ ]:
#pure numbers
for season in sorted(df_comp_lap["Season"].unique()):
    for gp in df_comp_lap[df_comp_lap["Season"].eq(season)]["GP"].unique():

        df_s_gp = df_comp_lap[
            (df_comp_lap["Season"].eq(season)) &
            (df_comp_lap["GP"].eq(gp))
        ]

        if df_s_gp.empty:
            continue

        for c in df_s_gp['Compound'].unique():
            print(f'{gp}-{season}-{c}')
            print(df_s_gp[df_s_gp['Compound'].eq(c)].groupby(['Driver','Session'])['LapTime'].describe())
            print('\n')

When looking at the graphics and descriptive statistics, we can observe a (linear) trend indicating that lap times generally decrease from P1 to Qualifying. P3 does not always continue this downward trend; in some cases, P3 stagnates or is even slower than P2. The same pattern can also be observed at driver level.

We can also observe that tyre compound has an impact on lap time. As expected, softer tyres generally produce faster laps. In later seasons, the naming convention changed: compounds such as Hypersoft and Ultrasoft were no longer used, and the visible labels were simplified to Soft, Medium and Hard. However, these labels can correspond to different underlying Pirelli compound specifications, ranging from C1 to C5 depending on the race weekend. Since we do not have access to the exact C-compound information for each tyre allocation, this should be addressed as a limitation of the project.

### How does Temperature-Driver/Team-GP-Season interact with each other

Next, we will check, how temperature, humidty, pressure and track temperature impact the lap time and the how the driver and teams differ.

#### AirTemperature

In [ ]:
for s in df['Season'].unique():
    for gp in df[df['Season'].eq(s)]['GP'].unique():
        race_weekend = df[(df['GP'].eq(gp)) & (df['Season'].eq(s))]

        plt.figure(figsize=(12,6))

        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p1', y='AirTemp_p1', label='P1')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p2', y='AirTemp_p2', label='P2')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p3', y='AirTemp_p3', label='P3')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_sprint_quali', y='AirTemp_sprint_quali', label='Sprint-Quali')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_quali', y='AirTemp_quali', label='Quali')


        p1_min_lap = race_weekend['laptime_sum_sectortimes_p1'].min()
        p2_min_lap = race_weekend['laptime_sum_sectortimes_p2'].min()
        p3_min_lap = race_weekend['laptime_sum_sectortimes_p3'].min()
        sprint_min_lap = race_weekend['laptime_sum_sectortimes_sprint_quali'].min()
        quali_min_lap = race_weekend['laptime_sum_sectortimes_quali'].min()

        if pd.notna(p1_min_lap):
            plt.axvline(p1_min_lap)
            p1_airtemp = race_weekend[race_weekend['laptime_sum_sectortimes_p1'].eq(p1_min_lap)]['AirTemp_p1'].iloc[0]
            plt.axhline(p1_airtemp)
            print(f'{gp}-{s}: Fastest time in P1: {p1_min_lap}; AirTemp during fastest time in P1: {p1_airtemp}')

        if pd.notna(p2_min_lap):
            plt.axvline(p2_min_lap, c='orange')
            p2_airtemp = race_weekend[race_weekend['laptime_sum_sectortimes_p2'].eq(p2_min_lap)]['AirTemp_p2'].iloc[0]
            plt.axhline(p2_airtemp, c='orange')
            print(f'{gp}-{s}: Fastest time in P2: {p2_min_lap}; AirTemp during fastest time in P2: {p2_airtemp}')

        if pd.notna(p3_min_lap):
            plt.axvline(p3_min_lap, c='green')
            p3_airtemp = race_weekend[race_weekend['laptime_sum_sectortimes_p3'].eq(p3_min_lap)]['AirTemp_p3'].iloc[0]
            plt.axhline(p3_airtemp, c='green')
            print(f'{gp}-{s}: Fastest time in P3: {p3_min_lap}; AirTemp during fastest time in P3: {p3_airtemp}')

        if pd.notna(sprint_min_lap):
            plt.axvline(sprint_min_lap, c='red')
            sprint_airtemp = race_weekend[race_weekend['laptime_sum_sectortimes_sprint_quali'].eq(sprint_min_lap)]['AirTemp_sprint_quali'].iloc[0]
            plt.axhline(sprint_airtemp, c='red')
            print(f'{gp}-{s}: Fastest time in Sprint-Qualification: {sprint_min_lap}; AirTemp during fastest time in Sprint-Qualification: {sprint_airtemp}')

        if pd.notna(quali_min_lap):
            plt.axvline(quali_min_lap, c='purple')
            quali_airtemp = race_weekend[race_weekend['laptime_sum_sectortimes_quali'].eq(quali_min_lap)]['AirTemp_quali'].iloc[0]
            plt.axhline(quali_airtemp, c='purple')
            print(f'{gp}-{s}: Fastest time in Qualification: {quali_min_lap}; AirTemp during fastest time in Qualification: {quali_airtemp}\n')

        plt.legend(title='Session')
        plt.xlabel('LapTime')
        plt.ylabel('AirTemp')
        plt.title(f'Scatterplot AirTemp - LapTime for {gp}-{s}')

The plots suggest that air temperature may have an influence on lap time, but the relationship is not perfectly linear. In some cases, higher air temperatures coincide with faster lap times, possibly because warmer conditions help tyres reach their operating window. In other cases, higher temperatures coincide with slower lap times, which may indicate overheating or tyre degradation effects. Since the data only contains each driver’s best lap per session, the laps are performance-relevant, but important confounding factors such as tyre compound, fuel load, engine mode, setup, and track evolution are still not fully controlled. Therefore, air temperature should be treated as a contextual performance factor rather than an isolated explanatory variable.

#### Humidity

In [ ]:
for s in df['Season'].unique():
    for gp in df[df['Season'].eq(s)]['GP'].unique():
        race_weekend = df[(df['GP'].eq(gp)) & (df['Season'].eq(s))]

        plt.figure(figsize=(12,6))

        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p1', y='Humidity_p1', label='P1')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p2', y='Humidity_p2', label='P2')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p3', y='Humidity_p3', label='P3')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_sprint_quali', y='Humidity_sprint_quali', label='Sprint-Quali')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_quali', y='Humidity_quali', label='Quali')


        p1_min_lap = race_weekend['laptime_sum_sectortimes_p1'].min()
        p2_min_lap = race_weekend['laptime_sum_sectortimes_p2'].min()
        p3_min_lap = race_weekend['laptime_sum_sectortimes_p3'].min()
        sprint_min_lap = race_weekend['laptime_sum_sectortimes_sprint_quali'].min()
        quali_min_lap = race_weekend['laptime_sum_sectortimes_quali'].min()

        if pd.notna(p1_min_lap):
            plt.axvline(p1_min_lap)
            p1_hum = race_weekend[race_weekend['laptime_sum_sectortimes_p1'].eq(p1_min_lap)]['Humidity_p1'].iloc[0]
            plt.axhline(p1_hum)
            print(f'{gp}-{s}: Fastest time in P1: {p1_min_lap}; Humidity during fastest time in P1: {p1_hum}')

        if pd.notna(p2_min_lap):
            plt.axvline(p2_min_lap, c='orange')
            p2_hum = race_weekend[race_weekend['laptime_sum_sectortimes_p2'].eq(p2_min_lap)]['Humidity_p2'].iloc[0]
            plt.axhline(p2_hum, c='orange')
            print(f'{gp}-{s}: Fastest time in P2: {p2_min_lap}; Humidity during fastest time in P2: {p2_hum}')

        if pd.notna(p3_min_lap):
            plt.axvline(p3_min_lap, c='green')
            p3_hum = race_weekend[race_weekend['laptime_sum_sectortimes_p3'].eq(p3_min_lap)]['Humidity_p3'].iloc[0]
            plt.axhline(p3_hum, c='green')
            print(f'{gp}-{s}: Fastest time in P3: {p3_min_lap}; Humidity during fastest time in P3: {p3_hum}')

        if pd.notna(sprint_min_lap):
            plt.axvline(sprint_min_lap, c='red')
            sprint_hum = race_weekend[race_weekend['laptime_sum_sectortimes_sprint_quali'].eq(sprint_min_lap)]['Humidity_sprint_quali'].iloc[0]
            plt.axhline(sprint_hum, c='red')
            print(f'{gp}-{s}: Fastest time in Sprint-Qualification: {sprint_min_lap}; Humidity during fastest time in Sprint-Qualification: {sprint_hum}')

        if pd.notna(quali_min_lap):
            plt.axvline(quali_min_lap, c='purple')
            quali_hum = race_weekend[race_weekend['laptime_sum_sectortimes_quali'].eq(quali_min_lap)]['Humidity_quali'].iloc[0]
            plt.axhline(quali_hum, c='purple')
            print(f'{gp}-{s}: Fastest time in Qualification: {quali_min_lap}; Humidity during fastest time in Qualification: {quali_hum}\n')

        plt.legend(title='Session')
        plt.xlabel('LapTime')
        plt.ylabel('Humidity')
        plt.title(f'Scatterplot Humidity - LapTime for {gp}-{s}')

Compared with air temperature, humidity shows an even weaker visible relationship with lap time. While some high-humidity sessions are associated with slower laps, this is usually also linked to wet or changing conditions rather than humidity alone. Humidity seems to be not strong enough, to be a stand alone feature, it might performe better in combination with other features.

#### Pressure

In [ ]:
for s in df['Season'].unique():
    for gp in df[df['Season'].eq(s)]['GP'].unique():
        race_weekend = df[(df['GP'].eq(gp)) & (df['Season'].eq(s))]

        plt.figure(figsize=(12,6))

        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p1', y='Pressure_p1', label='P1')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p2', y='Pressure_p2', label='P2')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p3', y='Pressure_p3', label='P3')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_sprint_quali', y='Pressure_sprint_quali', label='Sprint-Quali')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_quali', y='Pressure_quali', label='Quali')


        p1_min_lap = race_weekend['laptime_sum_sectortimes_p1'].min()
        p2_min_lap = race_weekend['laptime_sum_sectortimes_p2'].min()
        p3_min_lap = race_weekend['laptime_sum_sectortimes_p3'].min()
        sprint_min_lap = race_weekend['laptime_sum_sectortimes_sprint_quali'].min()
        quali_min_lap = race_weekend['laptime_sum_sectortimes_quali'].min()

        if pd.notna(p1_min_lap):
            plt.axvline(p1_min_lap)
            p1_pre = race_weekend[race_weekend['laptime_sum_sectortimes_p1'].eq(p1_min_lap)]['Pressure_p1'].iloc[0]
            plt.axhline(p1_pre)
            print(f'{gp}-{s}: Fastest time in P1: {p1_min_lap}; Pressure during fastest time in P1: {p1_pre}')

        if pd.notna(p2_min_lap):
            plt.axvline(p2_min_lap, c='orange')
            p2_pre = race_weekend[race_weekend['laptime_sum_sectortimes_p2'].eq(p2_min_lap)]['Pressure_p2'].iloc[0]
            plt.axhline(p2_pre, c='orange')
            print(f'{gp}-{s}: Fastest time in P2: {p2_min_lap}; Pressure during fastest time in P2: {p2_pre}')

        if pd.notna(p3_min_lap):
            plt.axvline(p3_min_lap, c='green')
            p3_pre = race_weekend[race_weekend['laptime_sum_sectortimes_p3'].eq(p3_min_lap)]['Pressure_p3'].iloc[0]
            plt.axhline(p3_pre, c='green')
            print(f'{gp}-{s}: Fastest time in P3: {p3_min_lap}; Pressure during fastest time in P3: {p3_pre}')

        if pd.notna(sprint_min_lap):
            plt.axvline(sprint_min_lap, c='red')
            sprint_pre = race_weekend[race_weekend['laptime_sum_sectortimes_sprint_quali'].eq(sprint_min_lap)]['Pressure_sprint_quali'].iloc[0]
            plt.axhline(sprint_pre, c='red')
            print(f'{gp}-{s}: Fastest time in Sprint-Qualification: {sprint_min_lap}; Pressure during fastest time in Sprint-Qualification: {sprint_pre}')

        if pd.notna(quali_min_lap):
            plt.axvline(quali_min_lap, c='purple')
            quali_pre = race_weekend[race_weekend['laptime_sum_sectortimes_quali'].eq(quali_min_lap)]['Pressure_quali'].iloc[0]
            plt.axhline(quali_pre, c='purple')
            print(f'{gp}-{s}: Fastest time in Qualification: {quali_min_lap}; Pressure during fastest time in Qualification: {quali_pre}\n')

        plt.legend(title='Session')
        plt.xlabel('LapTime')
        plt.ylabel('Pressure')
        plt.title(f'Scatterplot Pressure - LapTime for {gp}-{s}')

Humidity appears to have a weaker and less consistent relationship with lap time than air temperature. Across race weekends, there is no clear pattern where higher or lower humidity systematically leads to faster laps. In many cases, the fastest qualifying lap occurs at very different humidity levels, depending on the circuit, session, and weather context. Therefore, humidity is likely more of a contextual variable than a strong standalone performance driver.

#### TrackTemperatur

In [ ]:
for s in df['Season'].unique():
    for gp in df[df['Season'].eq(s)]['GP'].unique():
        race_weekend = df[(df['GP'].eq(gp)) & (df['Season'].eq(s))]

        plt.figure(figsize=(12,6))

        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p1', y='TrackTemp_p1', label='P1')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p2', y='TrackTemp_p2', label='P2')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_p3', y='TrackTemp_p3', label='P3')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_sprint_quali', y='TrackTemp_sprint_quali', label='Sprint-Quali')
        sns.scatterplot(data=race_weekend, x='laptime_sum_sectortimes_quali', y='TrackTemp_quali', label='Quali')


        p1_min_lap = race_weekend['laptime_sum_sectortimes_p1'].min()
        p2_min_lap = race_weekend['laptime_sum_sectortimes_p2'].min()
        p3_min_lap = race_weekend['laptime_sum_sectortimes_p3'].min()
        sprint_min_lap = race_weekend['laptime_sum_sectortimes_sprint_quali'].min()
        quali_min_lap = race_weekend['laptime_sum_sectortimes_quali'].min()

        if pd.notna(p1_min_lap):
            plt.axvline(p1_min_lap)
            p1_tracktemp = race_weekend[race_weekend['laptime_sum_sectortimes_p1'].eq(p1_min_lap)]['TrackTemp_p1'].iloc[0]
            plt.axhline(p1_tracktemp)
            print(f'{gp}-{s}: Fastest time in P1: {p1_min_lap}; TrackTemp during fastest time in P1: {p1_tracktemp}')

        if pd.notna(p2_min_lap):
            plt.axvline(p2_min_lap, c='orange')
            p2_tracktemp = race_weekend[race_weekend['laptime_sum_sectortimes_p2'].eq(p2_min_lap)]['TrackTemp_p2'].iloc[0]
            plt.axhline(p2_tracktemp, c='orange')
            print(f'{gp}-{s}: Fastest time in P2: {p2_min_lap}; TrackTemp during fastest time in P2: {p2_tracktemp}')

        if pd.notna(p3_min_lap):
            plt.axvline(p3_min_lap, c='green')
            p3_tracktemp = race_weekend[race_weekend['laptime_sum_sectortimes_p3'].eq(p3_min_lap)]['TrackTemp_p3'].iloc[0]
            plt.axhline(p3_tracktemp, c='green')
            print(f'{gp}-{s}: Fastest time in P3: {p3_min_lap}; TrackTemp during fastest time in P3: {p3_tracktemp}')

        if pd.notna(sprint_min_lap):
            plt.axvline(sprint_min_lap, c='red')
            sprint_tracktemp = race_weekend[race_weekend['laptime_sum_sectortimes_sprint_quali'].eq(sprint_min_lap)]['TrackTemp_sprint_quali'].iloc[0]
            plt.axhline(sprint_tracktemp, c='red')
            print(f'{gp}-{s}: Fastest time in Sprint-Qualification: {sprint_min_lap}; TrackTemp during fastest time in Sprint-Qualification: {sprint_tracktemp}')

        if pd.notna(quali_min_lap):
            plt.axvline(quali_min_lap, c='purple')
            quali_tracktemp = race_weekend[race_weekend['laptime_sum_sectortimes_quali'].eq(quali_min_lap)]['TrackTemp_quali'].iloc[0]
            plt.axhline(quali_tracktemp, c='purple')
            print(f'{gp}-{s}: Fastest time in Qualification: {quali_min_lap}; TrackTemp during fastest time in Qualification: {quali_tracktemp}\n')

        plt.legend(title='Session')
        plt.xlabel('LapTime')
        plt.ylabel('TrackTemp')
        plt.title(f'Scatterplot Pressure - LapTime for {gp}-{s}')

Track temperature appears to have no clear linear relationship with lap time when observed descriptively. In some race weekends, faster laps occur at higher track temperatures, while in others faster laps occur at lower or moderate track temperatures. This suggests that track temperature alone is not a strong standalone explanatory variable. Its effect is likely non-linear and strongly dependent on tyre compound, tyre operating window, track surface, circuit characteristics, session type and fuel load.

Next, we will focus on a statistical test to confirm statistical importance.

# Statistical Analyis

Here, we use a more statistical approach by checking feature distributions, correlations & monotonicity, outlier-check,multicollinearity, and hypothesis/ANOVA tests to assess whether the means of different groups differ from each other.

## Distribution/Testing for normality

In this section we will determine, if the data has a normal or non-normal distribution, we will do that with the D'Angostin and Anderson test.

- D'Angostino Test => analyses kurtosis and skewness of the data
- Anderson Test => analyses the tails/both ends of the distribution to see if the data is normal or non-normal

We will use these two test instead of Kolmogorow-Smirnow-Test, due to the "Central Limit Theorem", sampling distribution of the mean becomes approximately normal under certain conditions. This leads to, that Kolmogorow-Smirnow-Test can be very optimistic in determining normal distributions

In [ ]:
#getting the numerical columns
df_num = df.select_dtypes(exclude='object')
df_num_col = [c for c in df_num.columns if 'laptime_sum_sectortimes_quali' not in c and 'LapTimeDiff_quali' not in c]

#getting the object columns => we check distribution by counting appearance, which we did earlyer
df_ob = df.select_dtypes(exclude=[float,int])
df_ob_col = [c for c in df_ob.columns if 'Session' not in c]

#setting up the normal and non-normal lists for the D'Angostino Test
normal_lst_ango = set(n for n in df_num_col if normaltest(df_num[n].dropna(),nan_policy='omit')[1]> 0.05)
non_normal_lst_ango = set(cn for cn in df_num_col if cn not in normal_lst_ango)

#setting up the normal and non-normal lists for the Anderson Test
normal_lst_ander = set()
non_normal_lst_ander = set()

#calculating anderson score, to determin if the feature is normal or non-normal distributed
for c in df_num_col:
    data = df_num[c].dropna()
    result = anderson(data,dist='norm')
    try:
        index = result.significance_level.tolist().index(5.0)
        cv = result.critical_values[index]
        if result.statistic > cv:
            non_normal_lst_ander.add(c)
        else:
            normal_lst_ander.add(c)
    except ValueError:
        print(f'5% significance level not found for feature {c}')

print("D'Agostino Test")
print(f'Normally Distributed: {normal_lst_ango}')
print(f'Non-Normally Distributed: {non_normal_lst_ango}\n')

print("Anderson Test")
print(f'Normally Distributed: {normal_lst_ander}')
print(f'Non-Normally Distributed: {non_normal_lst_ander}')

In [ ]:
#checking distribution of y feature
stat_,p = normaltest(df['laptime_sum_sectortimes_quali'].dropna(),nan_policy='omit')
print(f"{'y feature is non-normally distributed' if p < 0.05 else 'y feature is normally distributed'}")

Just determining the if a feature is normally distributed, for model selection it is more important to check if the residuals of the X features and y feature are normally distributed. I will analyse each X feature's residual seperately and then the residuals of all X features together.

In [ ]:
resid_normal = []
resid_non_normal = []

for nc in df_num_col:
    y_feat = 'laptime_sum_sectortimes_quali'
    X_feat = nc

    df_tmp = df[[y_feat,X_feat]].dropna().copy()
    X_ = sm.add_constant(df_tmp[X_feat])
    y_ = df_tmp[y_feat]

    model = sm.OLS(y_,X_).fit()
    residuals = model.resid

    stat,p_value = normaltest(residuals)
    if p_value < 0.05:
        resid_non_normal.append(nc)
    else:
        resid_normal.append(nc)
print(f"Feature with normal residuals: {resid_normal}")
print(f"Feature with non-normal residuals: {resid_non_normal}")

In [ ]:
y_feat = "laptime_sum_sectortimes_quali"

X_cols = [c for c in df_num_col if c != y_feat]
sub_cols = X_cols + [y_feat]

df_tmp = df[sub_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

print("Original rows:", len(df))
print("Rows after dropna:", len(df_tmp))
print("Number of X features:", len(X_cols))

if len(df_tmp) < 8:
    print("Not enough complete rows for OLS/residual normality test.")
else:
    X_ = sm.add_constant(df_tmp[X_cols])
    y_ = df_tmp[y_feat]

    model = sm.OLS(y_, X_).fit()
    residuals = model.resid

    stat, p_value = normaltest(residuals)

    print("p-value:", p_value)

    if p_value < 0.05:
        print("Residuals are non-normally distributed")
    else:
        print("No strong evidence against normal residuals")

The test fails due to too many dropped rows, therefore we will split into normal race weekends and sprint weekends.

In [ ]:
y_feat = "laptime_sum_sectortimes_quali"

X_cols = [
    c for c in df_num_col
    if c != y_feat and "sprint_quali" not in c
]

sub_cols = X_cols + [y_feat]

df_tmp = df[sub_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

print(df_tmp.shape)

X_ = sm.add_constant(df_tmp[X_cols])
y_ = df_tmp[y_feat]

model = sm.OLS(y_, X_).fit()
residuals = model.resid

stat, p_value = normaltest(residuals)

if p_value < 0.05:
    print("Residuals are non-normally distributed")
else:
    print("No strong evidence against normal residuals")

In [ ]:
y_feat = "laptime_sum_sectortimes_quali"

X_cols_sprint = [
    c for c in df_num_col
    if c != y_feat and "sprint_quali" in c
]

sub_cols_sprint = X_cols_sprint + [y_feat]

df_sprint = df[sub_cols_sprint].replace([np.inf, -np.inf], np.nan).dropna().copy()

print(df_sprint.shape)

if len(df_sprint) >= 8:
    X_ = sm.add_constant(df_sprint[X_cols_sprint])
    y_ = df_sprint[y_feat]

    model = sm.OLS(y_, X_).fit()
    residuals = model.resid

    stat, p_value = normaltest(residuals)

    if p_value < 0.05:
        print("Sprint model residuals are non-normally distributed")
    else:
        print("No strong evidence against normal sprint model residuals")
else:
    print("Not enough sprint rows.")

We can see that the majority of simple one-feature OLS residuals are non-normally distributed. This suggests that simple linear relationships may not fully describe the relationship between the individual features and qualifying lap time. Therefore, tree-based models should be considered as candidate models.

Next, we will check whether the features have linear, monotonic, or nonlinear relationships with the target. This will help us better understand whether linear models are reasonable candidates or whether nonlinear models such as tree-based models may be more suitable. The final model choice will still be based on cross-validation performance.

## Correlation & Linearity
### Correlation for numeric values

As we have established the majority of the data is non-normal distributed, we will use Pearson correlation for determining if there is a linear relationship between the y-feature ("laptime_sum_sectortimes_quali") and X features.

At the same time we will check if the X features do have a monotonic relationship, with spearman correlation.

These two aspects will support the decision making process of selecting the most appropriate model for the task.

In [ ]:
df_corr = pd.DataFrame()

for c in [c for c in df_num.columns if 'laptime_sum_sectortimes_quali' not in c and "LapTimeDiff_quali" not in c]:
    #print(c)
    x_col = c
    y_col = 'laptime_sum_sectortimes_quali'

    #here we add the two features together (X and y), so when we drop NaN, both features lose the same row, index length stays equal
    tmp = (
        df_num[[x_col, y_col]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    stat_p,p_p = pearsonr(x = tmp[x_col], y= tmp[y_col])
    stat_s,p_s = spearmanr(a = tmp[x_col], b= tmp[y_col])
    mi_score = mutual_info_regression(X=tmp[[x_col]],y=tmp[y_col],random_state=101)[0]

    #when p is smaller than 0.05 then the result is statistically significant
    #when the p is bigger than 0.05, then the result is not statistically significant
    sig_level_p = 'Sig' if p_p < 0.05 else 'not Sig'
    sig_level_s = 'Sig' if p_s < 0.05 else 'not Sig'
    df_loop = pd.DataFrame.from_dict(
        {
            'Feature Combo':f'{c}-laptime_sum_sectortimes_quali',
            'Correlation-Level': stat_p,
            'Stats_Sig_Pearson':sig_level_p,
            'Monotonic-Level':stat_s,
            'Stats_Sig_Spearman':sig_level_s,
            "Mutual Information Score": mi_score
        },orient='index'
    ).transpose()

    df_corr = pd.concat([df_corr,df_loop],axis=0)

df_corr.sort_values('Correlation-Level',ascending=False)

Here we can see that P1, P2 and P3 lap times have a strong relationship on the qualification lap time. Also the circut length has a strong relationship on the qualification lap time. When we look at the AirTemp and TrackTemp features, we can see, that they have a weak to moderate negative relationship on the the qualification. The last aspect, temp and qualification lap time will be anaylsed seperatly, is that a universally observable or number of turns and circut length depentable, I would do that with another correlation test.

When we look at the Mutual Information score, we can see that some features with a weak/medium correlation have a higher mutual information score, this is an indicator, that a tree-based model might be able to better pick up the signal of the data.

In [ ]:
target = "laptime_sum_sectortimes_quali"

temp_cols = [
    "TrackTemp_quali",
    "TrackTemp_sprint_quali",
    "TrackTemp_p1",
    "TrackTemp_p2",
    "TrackTemp_p3"
]

results = []

for t in temp_cols:

    # Skip if column does not exist
    if t not in df.columns:
        print(f"Column not found: {t}")
        continue

    for gp in df["GP"].dropna().unique():

        tmp = (
            df.loc[df["GP"].eq(gp), [t, target]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        n = len(tmp)

        # Pearson/Spearman need enough rows and variation
        if n < 3 or tmp[t].nunique() < 2 or tmp[target].nunique() < 2:
            results.append({
                "GP": gp,
                "Temp": t,
                "n": n,
                "Correlation-Level": np.nan,
                "Stats_Sig_Pearson": "not enough data",
                "Monotonic-Level": np.nan,
                "Stats_Sig_Spearman": "not enough data"
            })
            continue

        pearson_r, pearson_p = pearsonr(tmp[t], tmp[target])
        spearman_r, spearman_p = spearmanr(tmp[t], tmp[target])
        mi_score = mutual_info_regression(X=tmp[[t]],y=tmp[target],random_state=101)[0]

        results.append({
            "GP": gp,
            "Temp": t,
            "n": n,
            "Correlation-Level": pearson_r,
            "Stats_Sig_Pearson": "Sig" if pearson_p < 0.05 else "not Sig",
            "Monotonic-Level": spearman_r,
            "Stats_Sig_Spearman": "Sig" if spearman_p < 0.05 else "not Sig",
            "Mutual Information Score": mi_score
            
        })

df_gp_temp_corr = pd.DataFrame(results)

df_gp_temp_corr.sort_values(["GP", "Temp"])

When we break it down into the different indiviual GPs we can see that temperature does not always have a negative relationship between qualification lap time. Track, weather and tires seem to influence the TrackTemp and qualification lap time relationship, if it is going up or down.

Next, we will check the correlation level between binary X features and y feature, this will be done by using pointbiserial correlation.

### Correlation of binary features

In [ ]:
binary_cols = [c for c in df_num.columns if df_num[c].min() == 0 and df_num[c].max() == 1]

for b in binary_cols:
    df_num_bc = df_num[[b,'laptime_sum_sectortimes_quali']].dropna()
    stat,p = pointbiserialr(df_num_bc.iloc[:,0],df_num_bc.iloc[:,1])
    p_value = 'Sig' if p < 0.05 else 'Not Sig'
    print(f"Point Biserial for {b}: {stat}; {p_value}")

The binary categorical variables show statistically significant relationships with qualifying lap time, but the effect sizes are weak. Therefore, although these variables may contain some information, their standalone explanatory power appears limited.

Next we will use LOWLESS Line to determine if a linear or tree-based model migt be more suiting.

### LOWLESS Curve

In [ ]:
for c in [nc for nc in df_num_col if df_num[nc].max() > 1]:
    y_feat = 'laptime_sum_sectortimes_quali'
    X_feat = c

    df_tmp = df[[X_feat,y_feat]].dropna().copy()
    plt.figure(figsize=(12,6))

    sns.scatterplot(
        data = df_tmp,
        x=X_feat,
        y=y_feat,
        alpha = 0.4
    )

    sns.regplot(
        data=df_tmp,
        x=X_feat,
        y=y_feat,
        scatter=False,
        color='blue',
        label = 'Linear'
    )
    plt.title(f"LOWESS Line for {X_feat} and {y_feat}")

With the LOWESS Line, we get a visual output to see, if the numerical features. Here we can observe the same as in my correlation analysis.

Next, we will run the same for the categorical values, here we will use boxplots, to see if there is shift/drift in distribution or not between the different categorical values.

### Linearity for Categorical Values

In [ ]:
for cc in df_ob_col:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df,x=cc,y='laptime_sum_sectortimes_quali')

As shown in the boxplots, some categorical features show clear shifts in the distribution of qualifying lap time across their category levels. For other categorical features, the distributional shift is less obvious, but the median or mean still differs between groups. This suggests that several categorical features may contain useful predictive information. After appropriate encoding, a tree-based model may be suitable because it can capture non-linear relationships and interactions between categorical and numerical features.

For nominal categorical variables with more than two groups, correlation or linearity cannot be measured directly because the categories do not have a natural numerical scale. Instead, we use group comparison methods to assess whether the mean qualifying lap time differs across category levels.

First, we analyze one categorical feature at a time using One-Way ANOVA. This allows us to test whether qualifying lap time differs significantly between the groups of a single categorical variable. After that, we use Two-Way ANOVA to analyze qualifying lap time under two categorical features simultaneously. This allows us to test the individual association of each categorical feature with lap time, as well as whether there is an interaction between both features.

## ANOVA
### One-Way ANOVA

Here will compare the all categorical values seperately with the qualification lap time, to confirm the boxplot visualization, that between the different categorical values there is a statistical significant difference in lap times. After that we will use a Two-Way or factorial ANOVA to see if the interaction of the different categorical values effect the lap time differently.

First we have to check the assumptions
- Normality Assumption => residulas need to be normally distributed
- Independet Observation => violated because each GP weekend contains multiple driver observations under the same circuit, weather, and session conditions
- Equal Variance => needs to be tested via levene test

#### Normality Test of Residuals

In [ ]:
df_ob_cols = df_ob.columns

target = 'laptime_sum_sectortimes_quali'

for cc in df_ob_col:
    factor = cc
    model = ols(f'Q("{target}") ~ C(Q("{factor}"))',data=df).fit()
    residuals = model.resid.dropna()
    if len(residuals) >= 8:
        stat,p = normaltest(residuals)
        test_name = "D'Agostino normaltest"
    else:
        stat,p = np.nan,np.nan
    if p < 0.05:
        print(f'{cc}-Residuals are non-normally distributed')
    else:
        print(f'{cc}-Residuals are normally distributed')

#### Levene's Test

In [ ]:
from scipy.stats import levene

variance_results = []

for c in df_ob_cols:
    tmp = (
        df[[c, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    groups = [
        group[target].values
        for _, group in tmp.groupby(c)
        if len(group[target].values) > 1
    ]

    if len(groups) < 2:
        continue

    stat, p = levene(*groups, center="median")

    variance_results.append({
        "feature": c,
        "n_groups": len(groups),
        "n_obs": len(tmp),
        "levene_stat": stat,
        "p_value": p,
        "equal_variance_assumption": "violated" if p < 0.05 else "not violated"
    })

df_variance_results = pd.DataFrame(variance_results)
df_variance_results

- Normality assumption: violated
- Independent observations: violated
- Equal variance assumption: violated

Because the normality and equal variance assumptions of standard ANOVA are violated, we use the Kruskal-Wallis test as a non-parametric alternative to One-Way ANOVA. If the Kruskal-Wallis test is statistically significant, Dunn’s post-hoc test is used to identify which groups differ from each other.

However, Kruskal-Wallis still assumes independent observations. Since the dataset contains repeated observations within GP weekends and repeated drivers/teams across seasons, the results should be interpreted as exploratory rather than as final causal/statistical proof.

#### Kruskal Wallis + Dunn's Test

In [ ]:
strong_es_cols = []
target = 'LapTimeDiff_quali'

#we exclude here GP, because we are well aware that the times differ from track to track
for c in df_ob_cols:
    df_krus = pg.kruskal(data=df,dv=target,between=c)
    
    #setting up and calculating effect size of Kruskal test
    tmp = df[[c,target]].dropna()
    H = df_krus["H"].iloc[0]
    p = df_krus["p_unc"].iloc[0]
    k = tmp[c].nunique()
    n = len(tmp)

    epsilon_sq = (H - k + 1) / (n - k)
    if epsilon_sq > 0.1:
        strong_es_cols.append(c)
    print(f"Effect Size for {c}: {epsilon_sq}")

    if df_krus['p_unc'].iloc[0] < 0.05:
        p_values = sp.posthoc_dunn(tmp,val_col=target,group_col=c, p_adjust='holm')
        print(f"{c}")
        df_sig_ov = p_values < 0.05
        sig_diff_lst = []
        for i in df_sig_ov.columns:
            sig_diff_lst.append({
                f'{c}':i,
                'Times_of_Sig_Diff':df_sig_ov[i].sum()
            })
            #print(f"{i}: {df_sig_ov[i].sum()}")
        sig_diff_df = pd.DataFrame(sig_diff_lst)
        print(sig_diff_df.sort_values('Times_of_Sig_Diff',ascending=False))
        #print(f"{p_values <0.05}\n")
    else:
        print(f"{c} - No significant difference between group values, Dunn's Test is not necessary\n")

print(f"\n Columsn with the strongest effect size: {strong_es_cols}")

Instead of using the absolute qualifying lap times of each driver across the 2018–2026 seasons, we use the qualifying lap-time difference to the fastest driver. This target variable is more suitable for performance comparison because it reduces the influence of circuit length and focuses on relative qualifying performance within each session.

The Kruskal-Wallis test is used to assess whether the distribution of qualifying lap-time differences differs significantly across the groups of each categorical feature. If a statistically significant difference is found, Dunn’s post-hoc test is applied to identify which specific category levels differ from each other.

For each categorical feature, we count how often a category level is significantly different from other levels. This provides an indication of how distinct a category is compared to the others. However, this count alone does not indicate whether a team, driver, compound, or other category is faster or slower. Another important note, is that the effect size of the Kruskal test is 0.29, which can be considered as a strong effect.

To obtain the direction of the relationship, we compare the mean or median LapTimeDiff_quali of each category level. Lower values indicate that the category is generally closer to the fastest qualifying lap time, while higher values indicate a larger gap. This additional comparison is only performed for features that showed statistically significant group differences in the Kruskal-Wallis test.

In [ ]:
for sc in strong_es_cols:
    overall_mean = df['LapTimeDiff_quali'].mean()
    overall_median = df['LapTimeDiff_quali'].median()

    feature_cat_value = [{
        f'{sc}':v,
        'Status_Mean': 'Faster' if df[df[sc].eq(v)]['LapTimeDiff_quali'].mean() < df['LapTimeDiff_quali'].mean() else 'Slower',
        'Status_Median': 'Faster' if df[df[sc].eq(v)]['LapTimeDiff_quali'].median() < df['LapTimeDiff_quali'].median() else 'Slower'
    } for v in df[sc].unique()]

    df_fast_slow = pd.DataFrame(feature_cat_value)
    print(f"{sc}")
    print(df_fast_slow)
    print('\n')

For the strongest categorical features identified by the Kruskal-Wallis effect size, we compare each category level with the overall mean and median of LapTimeDiff_quali. Since lower LapTimeDiff_quali values indicate a smaller gap to the fastest qualifying lap, category levels below the overall mean or median are classified as relatively faster, while values above the overall mean or median are classified as relatively slower.

This comparison does not replace the statistical group tests, but it adds directional interpretation. The Kruskal-Wallis and Dunn’s tests indicate whether statistically significant differences exist between groups, while the mean/median comparison helps identify whether a category tends to be associated with faster or slower relative qualifying performance.

For some drivers, the mean and median classification differ. This may indicate skewness, outliers, uneven sample sizes, team changes, regulation changes, or different career phases. Therefore, these cases should be interpreted carefully and may require additional season-level or team-level analysis.

The results also support expected patterns: stronger teams and later qualifying sessions tend to be associated with lower LapTimeDiff_quali values. Afterwards, a Two-Way ANOVA or multi-factor regression-style analysis will be used to investigate selected interaction effects, such as Driver × Tyre Compound,Driver x Season, Driver × Team, Team × Session, or Session × GP.

### Multifactor ANOVA & Two Way ANOVA

In this section we will perform a Multifactor ANOVA & Two-Way ANOVA. We will start with a Two-Way ANOVA. Here we will analyses different categorical features based on one continous feature. As alreaded stated, the categorical combinations are:
- Driver × Tyre Compound
- Driver x Season
- Driver × Team
- Team × Session
- Session × GP

If needed, we will perform a MulitFacor ANOVA, where we will compare more than two categorical based on the one continous value-

#### Two-Way ANOVA
The assumption of indepdent observation is violated, which will is giving already a big indicator to use a mixed-model, but we will perform the equval variance and normality test of the residuals, because they are different from the creteria for the One-Way ANOVA.

##### Equal Variance

In [ ]:
target_feature = 'LapTimeDiff_quali'
cats_combo = [
    ['Driver','Compound_quali'],['Driver','Season'],['Driver','Team'],['Team','Session'],['Session','GP']
]

for combo in cats_combo:
    tmp = (df[[target,combo[0],combo[1]]].replace([np.inf,-np.inf],np.nan).dropna())
    tmp['combined_group'] = tmp[combo[0]].astype(str)+'_'+tmp[combo[1]].astype(str)
    groups = [
        group[target].values for _,group in tmp.groupby('combined_group') if len(group) >= 2
    ]

    stat,p = levene(*groups,center='median')
    print(f"{combo[0]}-{combo[1]}: {'Equal variance violated' if p < 0.05 else 'Equal variance not violated'}")

#### Normality check of Residuals of categorical feature combination

In [ ]:
target_feature = 'LapTimeDiff_quali'
cats_combo = [
    ['Driver','Compound_quali'],['Driver','Season'],['Driver','Team'],['Team','Session'],['Session','GP']
]

for combo in cats_combo:
    tmp = (df[[target,combo[0],combo[1]]].replace([np.inf,-np.inf],np.nan).dropna())
    model = smf.ols(f"{target} ~ C({combo[0]}) * C({combo[1]})",data=tmp).fit()
    residuals = model.resid

    stat,p = normaltest(residuals)
    print(f"{combo[0]}-{combo[1]}: {'Residual normality violated' if p < 0.05 else 'Residual normality not violated'}")

Since the assumptions of normality, homogeneity of variance, and independence are violated, but a mixed-effects model can be difficult to estimate reliably because of sparse/unbalanced categorical combinations, we will use a Two-Way ANOVA, because it will work primarly as an exploratory analysis. For statistically significant feature interactions, Games-Howell simple-effects post-hoc tests are performed to identify which specific group pairs differ. Games-Howell is chosen because it is more robust to unequal variances and unequal group sizes than Tukey HSD. However, because the independence assumption is still problematic, the post-hoc results are interpreted as exploratory.

In [ ]:
sig_combo = []
for combo in [['Driver','Compound_quali'],['Driver','Season'],['Driver','Team'],['Team','Session'],['Session','GP']]:
    tmp = (df[[target,combo[0],combo[1]]].replace([np.inf,-np.inf],np.nan).dropna())
    model = smf.ols(f"{target} ~ C({combo[0]}) * C({combo[1]})",data=tmp).fit()
    #type = 3 => not equal number of interaction of the different unique values of each categorical feature
    two_way_anova = sm.stats.anova_lm(model,typ=3)
    two_way_anova.reset_index(inplace=True)
    if two_way_anova.iloc[3,-1] < 0.05:
        sig_combo.append([combo[0],combo[1]])

for sc in sig_combo:
    tmp = (df[[target,sc[0],sc[1]]].replace([np.inf,-np.inf],np.nan).dropna())
    pair_g = games_howell_simple_effects(
        data=tmp,
        dv=target,
        compare_factor=sc[0],
        within_factor = sc[1]
    )
    print(pair_g)

# Outlier

We have established that the majority of the features are not normally distributed. Based on that, we will use IQR as outlier detection method.

In [ ]:
#IQR to determine outliers
df_num_non_binary = [c for c in df_num_col if df_num[c].max() > 1]

outlier_lst = set()
for c in df_num_non_binary:
    #replacing NaN values with 
    s = df_num[c].replace([np.inf, -np.inf], np.nan).dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    IQR = q3 - q1

    lower_bound = q1 - 1.5*IQR
    upper_bound = q3 + 1.5*IQR

    outliers = len(s[(s<lower_bound) | (s>upper_bound)])
    if outliers > 0:
        outlier_lst.add(c)
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df_num,y=c)
    plt.title(f"Boxplot of {c}")
print("##################### Checking X features for outliers #####################")
print(outlier_lst)
print(f"Percentage of columns with outliers: {np.round((len(outlier_lst)/len(df_num_non_binary))*100,2)}%")
print(f"Number of features with outliers: {len(outlier_lst)}\n")

#robust Z score
robust_z_score_lst = set(
    c for c in df_num_non_binary if np.sum(np.abs(robust_z_score(df_num[c].replace([np.inf,-np.inf],np.nan).dropna())) > 3.5) > 0)
print(robust_z_score_lst)
print(f"Percentage of columns with outliers: {np.round((len(robust_z_score_lst)/len(df_num_non_binary))*100,2)}%")
print(f"Number of features with outliers: {len(robust_z_score_lst)}\n")

#columns which they do not have in common
print(f"Column(s) which both approaches do not have incommon: {outlier_lst.symmetric_difference(robust_z_score_lst)}\n")

print("##################### Checking y feature for outliers #####################")
s = df_num["laptime_sum_sectortimes_quali"].replace([np.inf, -np.inf], np.nan).dropna()
q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
IQR = q3 - q1

lower_bound = q1 - 1.5*IQR
upper_bound = q3 + 1.5*IQR

outliers = len(s[(s<lower_bound) | (s>upper_bound)])
if outliers > 0:
    print("According to IQR: y feature contains outliers")
else:
    print("According to IQR: y feature does not contain outliers")

robust_z_score_ = np.sum(np.abs(robust_z_score(s.replace([np.inf,-np.inf],np.nan).dropna())) > 3.5)
if robust_z_score_ > 0:
    print("According to Robust Z-Score: y feature contains outliers")
else:
    print("According to Robust Z-Score: y feature does not contain outliers")
plt.figure(figsize=(12,6))
sns.boxplot(data=df,y="laptime_sum_sectortimes_quali")
plt.title(f"Boxplot y feature")

Half of the X features contain outliers and y contains outliers as well. We will keep the outliers in our data, this is an important aspect, we need to consider for model selection later on.

Next, we will check for multicollinearity, if the X features correlate on themselves, this is important for the feature selection model and overall model.

# Multicollinearity
## Correlation Heatmap

In [ ]:
drop_cols = [
    'laptime_sum_sectortimes_quali','LapTimeDiff_quali','Sprint_Weekend','Rainfall_p1',
    'Rainfall_p2','Rainfall_p3','Rainfall_sprint_quali','Rainfall_quali','gp_id'
]

df_num_corr = df_num.drop(drop_cols,axis=1)
plt.figure(figsize=(26,20))
sns.heatmap(df_num_corr.corr(),annot=True)

## Variance Inflation factor

In [ ]:
sprint_week_vif = df_num_corr[
    (~df_num_corr["AirTemp_sprint_quali"].isna()) &
    (~df_num_corr["Humidity_sprint_quali"].isna()) &
    (~df_num_corr["Pressure_sprint_quali"].isna()) &
    (~df_num_corr["TrackTemp_sprint_quali"].isna()) &
    (~df_num_corr["laptime_sum_sectortimes_sprint_quali"].isna()) &
    (~df_num_corr["LapTimeDiff_sprint_quali"].isna()) 
].drop(
    [
        'AirTemp_p2','Humidity_p2','Pressure_p2','TrackTemp_p2','laptime_sum_sectortimes_p2','LapTimeDiff_p2',
        'AirTemp_p3','Humidity_p3','Pressure_p3','TrackTemp_p3','laptime_sum_sectortimes_p3','LapTimeDiff_p3'
    ],axis=1
)

reg_race_week_vif = df_num_corr[
    (df_num_corr["AirTemp_sprint_quali"].isna()) &
    (df_num_corr["Humidity_sprint_quali"].isna()) &
    (df_num_corr["Pressure_sprint_quali"].isna()) &
    (df_num_corr["TrackTemp_sprint_quali"].isna()) &
    (df_num_corr["laptime_sum_sectortimes_sprint_quali"].isna()) &
    (df_num_corr["LapTimeDiff_sprint_quali"].isna()) 
].drop(
    [
        'AirTemp_sprint_quali','Humidity_sprint_quali','Pressure_sprint_quali','TrackTemp_sprint_quali','laptime_sum_sectortimes_sprint_quali',
        'LapTimeDiff_sprint_quali'
    ],axis=1
)

In [ ]:
X = reg_race_week_vif.copy()

# replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# keep numeric columns only
X = X.select_dtypes(include=[np.number])

# remove columns that are fully missing
X = X.dropna(axis=1, how="all")

# remove constant columns
X = X.loc[:, X.nunique(dropna=True) > 1]

# impute remaining NaNs
X = X.fillna(X.median())

# final checks
print("Any NaN left:", X.isna().any().any())
print("Any inf left:", np.isinf(X.to_numpy()).any())
print("Shape:", X.shape)

x_vif = sm.add_constant(X, has_constant="add")

vif_df = pd.DataFrame()
vif_df["Features"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(x_vif.values, i)
    for i in range(1, x_vif.shape[1])
]
vif_df['Mulitcollinaity_Level'] = vif_df['VIF'].apply(lambda x: 'weak' if x < 5 else ('medium' if x <=10 and x >=5 else 'Strong'))

vif_df.sort_values("VIF", ascending=False)

In [ ]:
X = sprint_week_vif.copy()

# replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# keep numeric columns only
X = X.select_dtypes(include=[np.number])

# remove columns that are fully missing
X = X.dropna(axis=1, how="all")

# remove constant columns
X = X.loc[:, X.nunique(dropna=True) > 1]

# impute remaining NaNs
X = X.fillna(X.median())

# final checks
print("Any NaN left:", X.isna().any().any())
print("Any inf left:", np.isinf(X.to_numpy()).any())
print("Shape:", X.shape)

x_vif = sm.add_constant(X, has_constant="add")

vif_df = pd.DataFrame()
vif_df["Features"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(x_vif.values, i)
    for i in range(1, x_vif.shape[1])
]
vif_df['Mulitcollinaity_Level'] = vif_df['VIF'].apply(lambda x: 'weak' if x < 5 else ('medium' if x <=10 and x >=5 else 'Strong'))

vif_df.sort_values("VIF", ascending=False)

The VIF analysis shows that both regular race weekends and sprint race weekends contain multicollinearity, but the issue is especially pronounced for sprint weekends. This is mainly caused by structurally related feature groups, such as weather variables across sessions, lap-time variables across sessions, and circuit-layout variables. For example, variables such as Turns, Slow_turns, Medium_turns, and High_speed_turns are naturally dependent on each other, which explains the infinite VIF values. Similarly, pressure and temperature variables across sessions measure very similar race-weekend conditions.

This is an important insight for the modeling strategy. Since tree-based models are generally more robust to multicollinearity for prediction than linear models, multicollinearity should not necessarily be a major problem for model performance. However, it can still affect feature-importance interpretation, because importance may be split across correlated predictors. For feature selection and interpretation, permutation importance should preferably be applied with cross-validation and, where necessary, grouped by correlated feature families.

# Conclusion & next Steps

After the EDA/Descrpitive Analysis we have come to following conclusions.
- It makes sense to mean-encode Driver, Teams(-Lineage), Compunds, Complexity-Label, GP, Sprint-Race-Era and Pace Profile
    - Rare labels should be binned first and then encoded.
        - We will use separate RareLabelEncoders for Driver, GP, and all compound features.
        - Driver:
            - Categories with an appearance frequency below 0.0019 will be grouped.
            - Note: 0.0019 means 0.19%, and this will be the upper limit of the tested range.
        - GP:
            - Categories with an appearance frequency below 0.01 will be grouped.
            - Note: 0.01 means 1%, and this will be the upper limit of the tested range.
        - Compound features:
            - Categories with an appearance frequency below 0.015 will be grouped.
            - Note: 0.015 means 1.5%, and this will be the upper limit of the tested range.
        - For all mean-encoded features, we will also provide a frequency count, so the model can learn if mean value is based on several values or more shellow
        - This will be done by a Python class, which duplicates a feature and renames it
    - Rule Era will be mean- or ordinal- encoded
    - We will drop first Quali- and Sprint-Quali-Session, if the model needs it, we will add it again
- It makes sense to One-Hot encode (Race-Tack) Type and Pace-Profile
    - Regular Race Track => Track
    - Street => Street Circut
    - Pace-Profile => Five unique values
    - Rule Era => High Downforce, Ground Effect and Active Aero with Hybrid. Active Aero with Hybrid => will not be used in training, for the offset approach of the residuals, we set High Downforce and Ground Effect to zero, to indicate the model that something is different
- All Track Temperature features will be dropped, because it is very hard to an accurate value, right before the qualifing session, that we want to predict

Statistical Analysis
- The majority of the features are non-normal distributed, y feature is non-normal distributed
- The majority of the X feature's residuals are non-normally distributed (each X feature was indivually compared to y), and the overall residuals are non-normally distributed (all X features and y feature residuals)
- The majority of the data is non-linear and besides a few features the correlation is weak to medium, but the mutual information score is for more features high/relevent
    - The fact that mutual information score shows for some features stronger values than spearman or pearson, indicates that we need a model, which is able to deal with non-linear data
- Tyre compounds show relatively consistent monotonic differences in lap times, whereas most other categorical features exhibit complex and potentially non-linear relationships with the target
- The One-Way ANOVA and MultiFactor ANOVA we analysed if there is a statistical difference between one or two categorical values and lap time difference in qualifying
    - In the One-Way ANOVA, we found that Driver, Team and Team-Lineage have the stronges effect size, we could also find which categorical value was more often significantlly different to the other categorical values
    - In the Two-Way ANOVA, we compared two combinations categorical combinations under the numerical feature laptime difference in qualifying
    - Both analyses indicate that the categorical variables contain predictive information and therefore justify the use of target-based encoding techniques
- The IQR has indicated that we have to deal with outliers, they are ranging from small to bigger
- The Variance Inflation Factor and the correlation map of all X features indicated that we are dealing with multicollinarity from weak to strong
    - Since the data contains varying degrees of multicollinearity, permutation importance calculated on a Random Forest model will be used as the primary feature selection method

Based on all these aspects, it is better to use a tree-based model. We will attempt to use three different models.
- Random Forest Regression as baseline model
- Light GBM
- XGBoost